In [1]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

from forestvision.datasets import GNNForestAttr

# Feature Engineering  

Goal: Assess GNN class separability and reduce confusion in ODFW forest type classes

Steps
1. Extract mask with GNN and ODF clases and input data: Sentinel2, DEM, ClimateNA
   - Compute class frequencies per training tile.
   - Select tiles with high representation of confused classes.
   - Extract samples from selected tiles.
2. Perform LDA on extracted samples
3. Asess class separability with the J-M Distance Matrix
4. Remap GNN classes to reduce confusion in ODFW forest type classes

Goals

- Understand feature importance using LDA
- Understand and improve class separability
- Remap GNN classes to reduce confusion in ODFW forest type classes
- Assess class separability with the J-M Distance Matrix
- Assess classifier performance before and after remapping

In [2]:
paths = "../data/datasets/gnn/2021/"
gnn = GNNForestAttr(paths=paths, remap=False, res=10, crs="EPSG:5070")
gnn.is_image = True 

In [3]:
# data = np.loadtxt('../data/fortypba/data_table.csv', delimiter=',', skiprows=1)
data = pd.read_csv('../data/fortypba/data_table_s2loff.csv', header=0).sample(2000000)#, random_state=42)
data.shape

(2000000, 15)

In [4]:
data.columns = ["label"] + data.columns[1:].to_list()

In [5]:
data['label'] = data.label.astype(float).astype(int)

In [6]:
data.insert(1, 'odf', data.label.replace(gnn.remap_dict))
# data["odf"] = data.gnn.replace(gnn.remap_dict)

In [7]:
cl = [661, 721, 673, 945, 719]
# subset = data[data.gnn.isin(cl)][data.columns].drop("odf", axis=1).copy()
subset = data[data.odf.isin([10,11])][data.columns].drop("odf", axis=1).copy()
print(subset.shape)
subset.head()

(489275, 15)


,label,B1,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12,B13,centroid
3797877,714,76.0,47.0,55.0,17.0,102.0,507.0,737.0,647.0,743.0,1824.0,295.0,115.0,62.123203,6.813023e+17
21035689,902,147.5,221.0,363.0,317.0,717.0,1448.5,1741.0,1893.0,1905.5,2298.0,1264.5,804.5,125.020065,1.840944e+18
22282366,661,36.0,119.5,252.5,143.5,537.0,2014.5,2615.5,2899.0,2863.5,2215.5,699.0,293.0,66.458000,1.088465e+18
15025735,182,0.0,0.0,103.0,14.5,224.0,1257.5,1588.5,2085.0,1655.0,2143.5,508.0,210.5,600.994690,1.011156e+18
15118828,673,95.0,206.5,377.0,331.5,763.0,1557.5,1860.0,2143.5,2206.5,2228.0,1327.0,699.5,141.739517,1.016310e+18


In [8]:
data[data.odf.isin([10,11])]['odf'].value_counts()

odf
11    364823
10    124452
Name: count, dtype: int64

## LDA analysis

In [9]:
%load_ext cuml.accel

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

def perform_lda_analysis(df: pd.DataFrame, class_label: str):
    """
    Performs LDA on the given dataset and plots the results in 2D.

    Args:
        df (pd.DataFrame): The DataFrame containing pixel data.
        class_label (str): The name of the column containing class labels.
    """
    try:
        # Separate features (X) and target (y)
        X = df.drop(class_label, axis=1)
        y = df[class_label]
        unique_labels = sorted(y.unique())

        # Standardize the features (important for LDA)
        X_scaled = StandardScaler().fit_transform(X)

        # --- Perform LDA ---
        # We set n_components=2 to project the data into a 2D space for plotting
        lda = LinearDiscriminantAnalysis(n_components=3)
        X_lda = lda.fit_transform(X_scaled, y)
        
        # Create a DataFrame for plotting LDA results
        lda_df = pd.DataFrame(X_lda, columns=[f'LD{i+1}' for i in range(X_lda.shape[1])])
        lda_df[class_label] = y.values

        print(f"Explained variance ratio of the 2 linear discriminants: {lda.explained_variance_ratio_}")
 
       # --- 1. Explained Variance Ratio ---
        print("\n1. Explained Variance Ratio:")
        print("This shows the proportion of the inter-class variance explained by each discriminant.")
        for i, ratio in enumerate(lda.explained_variance_ratio_):
            print(f"  - Linear Discriminant {i+1} (LD{i+1}): {ratio:.2%}")
            
        # --- 2. Component Loadings (Coefficients) ---
        print("\n2. LDA Component Loadings (Scalings):")
        print("This table shows the weight of each spectral band on the discriminant axes.")
        print("A high absolute value means the band is highly influential for that axis.")
        
        loadings_df = pd.DataFrame(
            lda.scalings_,
            index=X.columns,
            columns=[f'LD{i+1}' for i in range(len(lda.scalings_[0]))]
        )
        print(loadings_df.round(4))
        
        # --- Interpretation Helper ---
        print("\n--- Interpretation ---")
        for i in range(min(5, len(unique_labels) - 1)):
            ld_name = f'LD{i+1}'
            # Get the band with the highest absolute loading for this component
            most_influential_band = loadings_df[ld_name].abs().idxmax()
            print(f"For {ld_name}, the most influential band is '{most_influential_band}'.")
        
        # Visualize the LDA separation
        # sns.scatterplot(data=lda_df, x='LD1', y='LD2', palette='viridis', hue=class_label, s=100)
        # plt.title("Sites Projected onto LDA Axes")
        # plt.show()

    except FileNotFoundError:
        print(f"Error: The file '{df}' was not found.")
    except KeyError:
        print(f"Error: The CSV must contain a '{class_label}' column.")
        raise
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        plt.close()
        return lda, lda_df


In [11]:
subset.label.unique()

array([714, 902, 661, 182, 673, 177, 919, 720, 670, 518, 948, 947, 945,
       607, 603, 677, 721, 685, 932, 942, 719, 701, 684, 581, 944, 681,
       926, 930, 186, 895, 893, 705, 718, 676, 602, 683, 600, 723,  35,
       898, 608, 911, 263, 667, 931, 894, 896, 887,  67, 690, 535, 906,
       689, 266, 889, 946, 935, 597, 717, 743, 272, 909, 269, 672, 908,
       888, 698, 921, 604, 598, 605, 886, 918, 425, 123, 679, 897, 601,
       916, 890, 262, 271, 599, 703, 891,  68, 170, 184, 910, 915, 704,
       368, 270, 580, 654, 899, 943, 901, 261, 900, 969, 892, 606, 286,
        43, 265,  69,  42])

In [12]:
lda, lda_res = perform_lda_analysis(subset, class_label='label');

Explained variance ratio of the 2 linear discriminants: [0.37023073 0.22126076 0.17435351]

1. Explained Variance Ratio:
This shows the proportion of the inter-class variance explained by each discriminant.
  - Linear Discriminant 1 (LD1): 37.02%
  - Linear Discriminant 2 (LD2): 22.13%
  - Linear Discriminant 3 (LD3): 17.44%

2. LDA Component Loadings (Scalings):
This table shows the weight of each spectral band on the discriminant axes.
A high absolute value means the band is highly influential for that axis.
             LD1     LD2     LD3     LD4     LD5     LD6     LD7     LD8  \
B1       -0.4422 -0.1723 -0.0205 -0.5198  0.6558  0.1364  0.9715 -0.3974   
B2        0.5387  0.4450 -1.1642  1.8768  1.3390 -0.2513  2.1717 -3.5929   
B3       -1.1642 -0.5933  2.6568 -2.1442 -1.6765  1.2490 -0.2253  8.4749   
B4        0.5495  0.2732 -1.1944 -0.0697 -0.0919 -0.4592 -0.9980 -6.7930   
B5       -0.2201 -0.4216 -1.3467  1.3455  1.9745 -2.0494 -2.3944  2.5950   
B6        0.6856  0.7237  1.

## Jeffries-Matusita (JM) distance matrix

The Jeffries-Matusita (JM) Distance is a statistical measure used to quantify the separability between two probability distributions, commonly applied in remote sensing and image classification to evaluate how distinct land cover types are based on their spectral signatures.

- The diagonal of `jm_matrix` must be zeros (distance from a class to itself)

- The matrix must be symmetric

- If `jm_matrix` is a pandas DataFrame, convert to numpy first: `squareform(jm_matrix.values)`


In [13]:
def calculate_jm_distance_matrix(df: str, class_label: str):
    """
    Calculates the Jeffries-Matusita (J-M) distance matrix between all pairs of classes.
    """
    try:
        X = df.drop(class_label, axis=1)
        y = df[class_label]
        class_labels = sorted(y.unique())
        jm_dist_matrix = pd.DataFrame(np.zeros((len(class_labels), len(class_labels))),
                                      index=class_labels, columns=class_labels)

        for i in range(len(class_labels)):
            for j in range(i, len(class_labels)):
                label_i = class_labels[i]
                label_j = class_labels[j]
                
                if i == j:
                    continue

                c_i = X[y == label_i].values
                c_j = X[y == label_j].values

                mean_i = np.mean(c_i, axis=0)
                cov_i = np.cov(c_i, rowvar=False)
                
                mean_j = np.mean(c_j, axis=0)
                cov_j = np.cov(c_j, rowvar=False)
                
                # Add a small regularization term to the diagonal to ensure invertibility
                cov_i += np.eye(cov_i.shape[0]) * 1e-6
                cov_j += np.eye(cov_j.shape[0]) * 1e-6

                mean_diff = mean_i - mean_j
                cov_avg = (cov_i + cov_j) / 2
                
                try:
                    cov_avg_inv = np.linalg.inv(cov_avg)
                    term1 = 0.125 * mean_diff.T @ cov_avg_inv @ mean_diff
                    
                    det_cov_avg = np.linalg.det(cov_avg)
                    det_cov_i = np.linalg.det(cov_i)
                    det_cov_j = np.linalg.det(cov_j)
                    
                    if det_cov_avg <= 0 or det_cov_i <= 0 or det_cov_j <= 0:
                        b_dist = 0 # Cannot compute, assume no separability
                    else:
                        term2 = 0.5 * np.log(det_cov_avg / np.sqrt(det_cov_i * det_cov_j))
                        b_dist = term1 + term2
                except np.linalg.LinAlgError:
                    b_dist = 0 # Cannot compute if matrix is singular

                jm_dist = 2 * (1 - np.exp(-b_dist)) # Jeffries-Matusita distance
                
                jm_dist_matrix.loc[label_i, label_j] = jm_dist
                jm_dist_matrix.loc[label_j, label_i] = jm_dist
                
        return jm_dist_matrix

    except FileNotFoundError:
        print(f"Error: The file '{df}' was not found.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred during J-M distance calculation: {e}")
        return None

In [14]:
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import warnings
from typing import Dict, Tuple, List, Callable


def compute_class_statistics(
    df: pd.DataFrame,
    class_column: str = 'class'
) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """
    Compute mean vector and covariance matrix for each class in the DataFrame.

    Args:
        df (pd.DataFrame): DataFrame with feature columns and a class label column.
            Shape: [N_samples, N_features + 1] where +1 is the class column.
        class_column (str): Name of the column containing class labels.

    Returns:
        Dict[str, Tuple[np.ndarray, np.ndarray]]: Dictionary mapping class labels
            to tuples of (mean, covariance) where:
            - mean: ndarray of shape [N_features]
            - covariance: ndarray of shape [N_features, N_features]

    Raises:
        ValueError: If class_column not found or no feature columns present.
    """
    if class_column not in df.columns:
        raise ValueError(f"Class column '{class_column}' not found in DataFrame. "
                        f"Available columns: {list(df.columns)}")

    feature_columns = [col for col in df.columns if col != class_column]
    if len(feature_columns) == 0:
        raise ValueError("No feature columns found in DataFrame.")

    class_stats = {}
    for label, group in df.groupby(class_column):
        n_samples = len(group)
        if n_samples < 2:
            warnings.warn(f"Class '{label}' has only {n_samples} sample(s). "
                         "Covariance will be regularized.")

        features = group[feature_columns].values
        mean = np.mean(features, axis=0)
        
        if n_samples >= 2:
            cov = np.cov(features, rowvar=False)
        else:
            cov = np.eye(len(feature_columns)) * 1e-3

        class_stats[label] = (mean, cov)

    return class_stats


def jeffries_matusita_metric(
    df: pd.DataFrame,
    class_column: str = 'class'
) -> Callable[[int, int], float]:
    """
    Create a custom JM distance metric function for use with pairwise_distances.

    Args:
        df (pd.DataFrame): Raw samples with feature columns and a class column.
            Shape: [N_samples, N_features + 1]
        class_column (str): Name of the column containing class labels.

    Returns:
        Callable[[int, int], float]: A metric function that takes indices i, j
            and returns JM distance in range [0, 2].
    """
    class_stats_dict = compute_class_statistics(df, class_column)

    class_labels = sorted(class_stats_dict.keys())
    means = [class_stats_dict[label][0] for label in class_labels]
    covs = [class_stats_dict[label][1] for label in class_labels]

    stats_lookup = {i: (means[i], covs[i]) for i in range(len(class_labels))}

    def jm_metric(i: int, j: int) -> float:
        """
        Compute JM distance between classes at indices i and j.

        Args:
            i (int): Index of first class.
            j (int): Index of second class.

        Returns:
            float: JM distance in range [0, 2].
        """
        mean1, cov1 = stats_lookup[i]
        mean2, cov2 = stats_lookup[j]

        if i == j:
            return 0.0

        n_features = len(mean1)
        cov1_reg = cov1 + np.eye(n_features) * 1e-6
        cov2_reg = cov2 + np.eye(n_features) * 1e-6

        pooled_cov = (cov1_reg + cov2_reg) / 2
        diff = mean1 - mean2

        try:
            inv_pooled_cov = np.linalg.inv(pooled_cov)
        except np.linalg.LinAlgError:
            inv_pooled_cov = np.linalg.pinv(pooled_cov)

        term1 = 0.125 * diff.T @ inv_pooled_cov @ diff

        det_cov1 = np.linalg.det(cov1_reg)
        det_cov2 = np.linalg.det(cov2_reg)
        det_pooled = np.linalg.det(pooled_cov)
        # det_sum = np.linalg.det(cov1_reg + cov2_reg)

        if det_cov1 <= 0 or det_cov2 <= 0 or det_pooled <= 0:
            warnings.warn("Non-positive determinant encountered. Using stronger regularization.")
            cov1_reg = cov1 + np.eye(n_features) * 1e-3
            cov2_reg = cov2 + np.eye(n_features) * 1e-3
            det_cov1 = np.linalg.det(cov1_reg)
            det_cov2 = np.linalg.det(cov2_reg)
            det_pooled = np.linalg.det(pooled_cov)

        term2 = 0.5 * np.log(det_pooled / (np.sqrt(det_cov1) * np.sqrt(det_cov2)))

        B = term1 + term2
        JM = 2 * (1 - np.exp(-B))

        return np.clip(JM, 0, 2)

    return jm_metric


def compute_jm_matrix_with_sklearn(
    df: pd.DataFrame,
    class_column: str = 'class'
) -> Tuple[np.ndarray, List]:
    """
    Compute JM distance matrix using sklearn's pairwise_distances approach.

    Args:
        df (pd.DataFrame): Raw samples with feature columns and a class column.
            Shape: [N_samples, N_features + 1]
        class_column (str): Name of the column containing class labels.

    Returns:
        Tuple[np.ndarray, List]:
            - jm_matrix: Symmetric distance matrix of shape [N_classes, N_classes].
              Values range from 0 (identical) to 2 (completely separable).
            - class_labels: Sorted list of class labels corresponding to matrix indices.

    Example:
        >>> df = pd.DataFrame({
        ...     'band1': [0.2, 0.3, 0.8, 0.9],
        ...     'band2': [0.1, 0.2, 0.7, 0.8],
        ...     'class': ['oak', 'oak', 'pine', 'pine']
        ... })
        >>> jm_matrix, labels = compute_jm_matrix_with_sklearn(df)
        >>> print(labels)
        ['oak', 'pine']
    """
    class_stats_dict = compute_class_statistics(df, class_column)

    class_labels = sorted(class_stats_dict.keys())
    n_classes = len(class_labels)

    indices = np.arange(n_classes).reshape(-1, 1)

    jm_metric_func = jeffries_matusita_metric(df, class_column)

    def metric_wrapper(x: np.ndarray, y: np.ndarray) -> float:
        """Wrapper to convert array indices to scalar indices."""
        i = int(x[0])
        j = int(y[0])
        return jm_metric_func(i, j)

    jm_matrix = pairwise_distances(indices, metric=metric_wrapper)

    return jm_matrix, class_labels 

In [15]:
def find_confused_classes(jm_dist_matrix: pd.DataFrame, threshold: float):
    """
    Identifies pairs of classes with a J-M distance below a given threshold.

    Args:
        jm_dist_matrix (pd.DataFrame): The pre-computed J-M distance matrix.
        threshold (float): The separability threshold. Pairs below this value
                            will be flagged as confused.

    Returns:
        A list of tuples, where each tuple contains the two class labels and
        their J-M distance.
    """
    confused_pairs = []
    # Use columns and index to iterate through pairs
    class_labels = jm_dist_matrix.columns
    
    # Iterate through the upper triangle of the matrix to avoid duplicates
    for i in range(len(class_labels)):
        for j in range(i + 1, len(class_labels)):
            label_i = class_labels[i]
            label_j = class_labels[j]
            
            dist = jm_dist_matrix.loc[label_i, label_j]
            
            if dist < threshold:
                confused_pairs.append((label_i, label_j, dist))
                
    # Sort by distance to see the most confused pairs first
    confused_pairs.sort(key=lambda x: x[2])
    
    return confused_pairs

In [16]:
from sklearn.preprocessing import StandardScaler

# Normalize the feature columns in subset (excluding 'label' class column)
feature_cols = subset.columns.drop('label')
scaler_subset = StandardScaler()
subset_normalized = subset.copy().reset_index(drop=True)
subset_normalized[feature_cols] = scaler_subset.fit_transform(subset[feature_cols])
print(subset_normalized.shape)
subset_normalized.head()

(489275, 15)


,label,B1,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12,B13,centroid
0,714,-0.159706,-0.297413,-0.463630,-0.391059,-0.771644,-1.598932,-1.622360,-1.741275,-1.818256,-0.610421,-1.160268,-0.951306,-0.876297,-0.943646
1,902,-0.042230,-0.008588,0.075441,0.142527,0.293251,-0.243635,-0.310749,-0.314259,-0.374679,0.066457,1.293821,1.718400,-0.709455,1.425291
2,661,-0.225427,-0.177069,-0.117960,-0.166064,-0.018425,0.571126,0.831684,0.837890,0.814953,-0.051354,-0.137625,-0.262100,-0.864799,-0.111886
3,182,-0.284576,-0.375429,-0.379619,-0.395506,-0.560396,-0.518581,-0.509973,-0.094366,-0.685746,-0.154171,-0.621102,-0.581535,0.553134,-0.269815
4,673,-0.128489,-0.032657,0.099944,0.168317,0.372902,-0.086729,-0.155289,-0.027367,-0.000901,-0.033504,1.452027,1.311846,-0.665104,-0.259286


In [17]:
# %%timeit
jm_matrix, class_labels = compute_jm_matrix_with_sklearn(lda_res, class_column='label')

In [18]:
jm_df = pd.DataFrame(jm_matrix, index=class_labels, columns=class_labels)
jm_df.head(20)

,35,42,43,67,68,69,123,170,177,182,...,932,935,942,943,944,945,946,947,948,969
35,0.000000,0.060641,0.329447,0.202525,0.342295,0.294866,0.218171,0.094124,0.650961,0.625636,...,0.897160,0.699042,0.886201,0.895554,0.906628,0.552381,0.912284,0.654368,0.769278,1.718779
42,0.060641,0.000000,0.225084,0.249715,0.274321,0.203937,0.238613,0.145609,0.641292,0.536126,...,0.711551,0.564077,0.798968,0.761039,0.808761,0.365725,0.757701,0.484869,0.624524,1.715001
43,0.329447,0.225084,0.000000,0.157414,0.056011,0.036766,0.766991,0.279099,1.261655,1.036461,...,0.815198,0.864086,1.220342,1.039457,1.174515,0.549941,0.936980,0.658959,0.833045,1.927368
67,0.202525,0.249715,0.157414,0.000000,0.186038,0.138782,0.683543,0.179912,1.166991,1.006143,...,0.972364,0.916548,1.214012,1.097521,1.190468,0.683312,1.032446,0.750319,0.911707,1.838537
68,0.342295,0.274321,0.056011,0.186038,0.000000,0.092425,0.744913,0.309547,1.273009,1.118826,...,0.860671,0.894019,1.268728,1.096205,1.214952,0.661405,0.977073,0.692902,0.835108,1.980825
69,0.294866,0.203937,0.036766,0.138782,0.092425,0.000000,0.718659,0.274176,1.242450,0.968278,...,0.762830,0.783165,1.170310,0.969414,1.118654,0.499759,0.865504,0.573255,0.742342,1.896240
123,0.218171,0.238613,0.766991,0.683543,0.744913,0.718659,0.000000,0.365332,0.283560,0.305938,...,0.830393,0.519773,0.585198,0.709417,0.627695,0.512855,0.796740,0.598744,0.635714,1.639173
170,0.094124,0.145609,0.279099,0.179912,0.309547,0.274176,0.365332,0.000000,0.827280,0.696052,...,0.923154,0.760686,0.925495,0.919277,0.918904,0.605907,0.931177,0.718506,0.813924,1.772815
177,0.650961,0.641292,1.261655,1.166991,1.273009,1.242450,0.283560,0.827280,0.000000,0.334951,...,0.954082,0.565313,0.409217,0.736435,0.565254,0.685285,0.936421,0.796860,0.799073,1.688016
182,0.625636,0.536126,1.036461,1.006143,1.118826,0.968278,0.305938,0.696052,0.334951,0.000000,...,0.568683,0.265603,0.217983,0.318752,0.249187,0.298288,0.504375,0.440017,0.440913,1.377813


In [19]:

print("\n" + "="*50)
print("JM Distance Matrix")
print("="*50)
print(f"\nClass labels: {len(class_labels)}")
print("\nJM Matrix:")
# print(np.round(jm_matrix, 2))

# Create labeled DataFrame for visualization
# jm_df = pd.DataFrame(jm_matrix, index=class_labels, columns=class_labels)
print("\nLabeled JM Distance Matrix:")
print(jm_df.round(2))

print("\n" + "="*50)

# Find min and max separability pairs
# Get the index (row, col) for the minimum non-zero value in the JM matrix
# Create a mask to ignore diagonal (zeros)
mask = jm_matrix > 0
min_val = jm_matrix[mask].min()
min_indices = np.where(jm_matrix == min_val)
min_pairs = [(class_labels[i], class_labels[j]) for i, j in zip(min_indices[0], min_indices[1])]
print(f"Min JM distance: {min_val:.4f} between classes: {min_pairs}")

max_val = jm_matrix.max()
max_indices = np.where(jm_matrix == max_val)
max_pairs = [(class_labels[i], class_labels[j]) for i, j in zip(max_indices[0], max_indices[1])]
print(f"Max JM distance: {max_val:.4f} between classes: {max_pairs}")

# Visualization
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Heatmap - convert to standard numpy to avoid cudf/cupy issues
# jm_array = np.array(jm_matrix, dtype=np.float64)
# sns.heatmap(jm_array, annot=True, fmt='.2f', cmap='RdYlGn', 
#             vmin=0, vmax=2, ax=axes[0], square=True,
#             xticklabels=class_labels, yticklabels="label")
# axes[0].set_title('JM Distance Matrix\n(0=identical, 2=completely separable)')

# Pairplot of features colored by class - use standard pandas if needed
# ax1 = axes[1]
# for label in class_labels:
#     subset = lda_res[lda_res['label'] == label]
#     # Convert to numpy to avoid cudf issues
#     x_vals = np.array(subset['LD1'].values, dtype=np.float64)
#     y_vals = np.array(subset['LD2'].values, dtype=np.float64)
#     ax1.scatter(x_vals, y_vals, label=label, alpha=0.6, s=30)
# ax1.set_xlabel('Linear Discriminant 1')
# ax1.set_ylabel('Linear Discriminant 2')
# ax1.set_title('Feature Space (LD1 vs LD2)')
# ax1.legend()

# plt.tight_layout()
# plt.show()


JM Distance Matrix

Class labels: 108

JM Matrix:

Labeled JM Distance Matrix:
      35    42    43    67    68    69    123   170   177   182  ...   932  \
35   0.00  0.06  0.33  0.20  0.34  0.29  0.22  0.09  0.65  0.63  ...  0.90   
42   0.06  0.00  0.23  0.25  0.27  0.20  0.24  0.15  0.64  0.54  ...  0.71   
43   0.33  0.23  0.00  0.16  0.06  0.04  0.77  0.28  1.26  1.04  ...  0.82   
67   0.20  0.25  0.16  0.00  0.19  0.14  0.68  0.18  1.17  1.01  ...  0.97   
68   0.34  0.27  0.06  0.19  0.00  0.09  0.74  0.31  1.27  1.12  ...  0.86   
..    ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   ...   
945  0.55  0.37  0.55  0.68  0.66  0.50  0.51  0.61  0.69  0.30  ...  0.20   
946  0.91  0.76  0.94  1.03  0.98  0.87  0.80  0.93  0.94  0.50  ...  0.06   
947  0.65  0.48  0.66  0.75  0.69  0.57  0.60  0.72  0.80  0.44  ...  0.12   
948  0.77  0.62  0.83  0.91  0.84  0.74  0.64  0.81  0.80  0.44  ...  0.11   
969  1.72  1.72  1.93  1.84  1.98  1.90  1.64  1.77  1.69  1.3

In [20]:
def get_most_similar_classes(jm_df: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    """
    For each class in the JM distance matrix, find the n most similar classes.

    Args:
        jm_df (pd.DataFrame): JM distance matrix with class labels as index and columns.
            Shape: [N_classes, N_classes]. Values range from 0 (identical) to 2 (separable).
        n (int): Number of most similar classes to return per class.

    Returns:
        pd.DataFrame: DataFrame with columns ['class', 'rank', 'similar_class', 'jm_distance'].
            Sorted by class and rank.
    """
    results = []
    
    for class_label in jm_df.index:
        # Get distances for this class, excluding self (distance = 0)
        distances = jm_df.loc[class_label].drop(class_label)
        
        # Sort by distance (ascending) and take top n
        most_similar = distances.nsmallest(n)
        
        for rank, (similar_class, distance) in enumerate(most_similar.items(), start=1):
            results.append({
                'class': class_label,
                'rank': rank,
                'similar_class': similar_class,
                'jm_distance': distance
            })
    
    return pd.DataFrame(results)

In [21]:
similar_classes = get_most_similar_classes(jm_df, n=5)

In [22]:
from sklearn.cluster import AgglomerativeClustering
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram

In [23]:
# Perform hierarchical clustering using complete linkage
# Convert square distance matrix to condensed form
jm_condensed = squareform(jm_matrix)
linked = linkage(jm_condensed, method='complete')

# Plot the dendrogram
# fig, ax = plt.subplots(figsize=(20, 14))
# dendrogram(linked, ax=ax)
# ax.tick_params(axis='both', labelsize=10)
# ax.set_title('Hierarchical Clustering Dendrogram', fontsize=18)
# plt.show()

# Extract clusters (e.g., cut at a distance threshold)
cluster = AgglomerativeClustering(n_clusters=3, metric='precomputed', linkage='complete')
labels = cluster.fit_predict(jm_matrix)
# print(labels)

In [24]:
clustered = pd.DataFrame({
    'cluster': labels
}, index=class_labels)#.sort_values(['cluster', 'class'])

clustered['odf'] = clustered.index.to_series().replace(gnn.remap_dict)
clustered

,cluster,odf
35,2,11
42,2,10
43,2,11
67,2,11
68,2,11
...,...,...
945,0,11
946,0,10
947,0,11
948,0,11


In [25]:
clustered.groupby('cluster').size()

cluster
0    67
1     1
2    40
dtype: int64

In [26]:
clustered_merged = clustered.merge(jm_df, left_index=True, right_index=True)
clustered_merged[clustered_merged.cluster == 0].head(50)

,cluster,odf,35,42,43,67,68,69,123,170,...,932,935,942,943,944,945,946,947,948,969
177,0,10,0.650961,0.641292,1.261655,1.166991,1.273009,1.242450,0.283560,0.827280,...,0.954082,0.565313,0.409217,0.736435,0.565254,0.685285,0.936421,0.796860,0.799073,1.688016
182,0,11,0.625636,0.536126,1.036461,1.006143,1.118826,0.968278,0.305938,0.696052,...,0.568683,0.265603,0.217983,0.318752,0.249187,0.298288,0.504375,0.440017,0.440913,1.377813
184,0,10,0.461973,0.390421,0.804311,0.775092,0.903331,0.727822,0.323468,0.494972,...,0.811885,0.567299,0.638968,0.647760,0.643852,0.424341,0.750446,0.602594,0.688623,1.369465
186,0,11,0.731900,0.709732,1.313299,1.230684,1.341739,1.297026,0.328992,0.869105,...,0.869231,0.492969,0.291101,0.612918,0.426177,0.644596,0.832776,0.758717,0.727935,1.667383
425,0,10,0.375834,0.225269,0.517466,0.585004,0.594166,0.456733,0.340811,0.498127,...,0.317893,0.176560,0.428650,0.342021,0.439914,0.060426,0.368212,0.143845,0.270479,1.574210
597,0,10,0.952945,0.935189,1.499169,1.354676,1.613138,1.458432,0.672052,1.066704,...,1.130571,0.728406,0.461270,0.765818,0.638960,0.775855,1.051328,0.983606,1.027618,1.317238
598,0,10,0.489262,0.536412,1.045015,0.823500,1.130639,1.008798,0.390850,0.528347,...,1.079568,0.740583,0.609974,0.831035,0.708942,0.661292,1.031436,0.884467,0.969462,1.482138
599,0,10,0.541462,0.524770,1.092283,0.966364,1.186264,1.045200,0.310045,0.605633,...,0.931074,0.567182,0.416482,0.638114,0.516795,0.525002,0.862346,0.748433,0.809171,1.383921
600,0,10,0.793737,0.811500,1.391228,1.231830,1.479240,1.367563,0.501842,0.906314,...,1.150047,0.783974,0.529294,0.853640,0.676325,0.803033,1.097727,1.013153,1.046013,1.627541
602,0,10,0.424142,0.378767,0.868912,0.789439,0.960542,0.809564,0.240090,0.471472,...,0.749244,0.430327,0.378033,0.520345,0.430654,0.357324,0.705765,0.563475,0.628249,1.437161


In [27]:
clustered_merged[clustered_merged.cluster == 2].head(50)

,cluster,odf,35,42,43,67,68,69,123,170,...,932,935,942,943,944,945,946,947,948,969
35,2,11,0.000000,0.060641,0.329447,0.202525,0.342295,0.294866,0.218171,0.094124,...,0.897160,0.699042,0.886201,0.895554,0.906628,0.552381,0.912284,0.654368,0.769278,1.718779
42,2,10,0.060641,0.000000,0.225084,0.249715,0.274321,0.203937,0.238613,0.145609,...,0.711551,0.564077,0.798968,0.761039,0.808761,0.365725,0.757701,0.484869,0.624524,1.715001
43,2,11,0.329447,0.225084,0.000000,0.157414,0.056011,0.036766,0.766991,0.279099,...,0.815198,0.864086,1.220342,1.039457,1.174515,0.549941,0.936980,0.658959,0.833045,1.927368
67,2,11,0.202525,0.249715,0.157414,0.000000,0.186038,0.138782,0.683543,0.179912,...,0.972364,0.916548,1.214012,1.097521,1.190468,0.683312,1.032446,0.750319,0.911707,1.838537
68,2,11,0.342295,0.274321,0.056011,0.186038,0.000000,0.092425,0.744913,0.309547,...,0.860671,0.894019,1.268728,1.096205,1.214952,0.661405,0.977073,0.692902,0.835108,1.980825
69,2,11,0.294866,0.203937,0.036766,0.138782,0.092425,0.000000,0.718659,0.274176,...,0.762830,0.783165,1.170310,0.969414,1.118654,0.499759,0.865504,0.573255,0.742342,1.896240
123,2,10,0.218171,0.238613,0.766991,0.683543,0.744913,0.718659,0.000000,0.365332,...,0.830393,0.519773,0.585198,0.709417,0.627695,0.512855,0.796740,0.598744,0.635714,1.639173
170,2,10,0.094124,0.145609,0.279099,0.179912,0.309547,0.274176,0.365332,0.000000,...,0.923154,0.760686,0.925495,0.919277,0.918904,0.605907,0.931177,0.718506,0.813924,1.772815
261,2,10,0.174275,0.193819,0.299082,0.211465,0.353868,0.299250,0.434252,0.066212,...,0.920748,0.785802,0.968373,0.928267,0.949240,0.573950,0.931849,0.712986,0.834250,1.729314
262,2,10,0.125990,0.095635,0.456578,0.435880,0.515822,0.428604,0.160504,0.156612,...,0.741381,0.501409,0.582303,0.649783,0.620999,0.363625,0.745014,0.545794,0.638361,1.631306


In [28]:
# %%timeit
# this is slower than sklearn version
# jm_matrix = calculate_jm_distance_matrix(lda_res, class_label='label')
# jm_matrix

In [29]:
# confused = find_confused_classes(jm_matrix, threshold=1.2)

In [30]:
lda_res.shape

(489275, 4)

In [31]:
lda_res['cluster'] = lda_res['label'].replace(clustered['cluster'].to_dict())
lda_res.head()

,LD1,LD2,LD3,label,cluster
0,-0.506790,0.479335,-0.755782,714,0
1,-1.407692,-2.393260,0.812483,902,0
2,0.039210,-0.065757,-1.037825,661,0
3,0.663450,0.733316,0.748839,182,0
4,-1.522577,-0.384829,-0.221222,673,0


## Classification 

In [32]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC, LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler


In [33]:
data = lda_res.copy()
data.shape

(489275, 5)

In [34]:
X, y = data.drop(['cluster', 'label'], axis=1), data['cluster'].astype(int)

In [35]:
# Normalize X using StandardScaler
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

# Convert the normalized array back to a DataFrame
X_normalized_df = pd.DataFrame(X_normalized, columns=X.columns)

# Display the first few rows of the normalized DataFrame
X_normalized_df.head()

,LD1,LD2,LD3
0,-0.455158,0.448311,-0.716458
1,-1.264276,-2.238365,0.770208
2,0.035216,-0.061501,-0.983826
3,0.595858,0.685855,0.709876
4,-1.367456,-0.359922,-0.209711


In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [37]:
clf = LinearSVC(penalty='l1', loss='squared_hinge', max_iter=10000)
clf.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l1'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [38]:
y_pred = clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.9115119309181953

In [39]:
print(classification_report(y_test, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.91      1.00      0.95     89211
           1       0.00      0.00      0.00        55
           2       0.49      0.03      0.06      8589

    accuracy                           0.91     97855
   macro avg       0.47      0.34      0.34     97855
weighted avg       0.88      0.91      0.87     97855

